# Problem Book P1, Chapter 13: Primes on Average (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the problems further, with
exercises first and solutions at the end. We use `numpy` for sieves, `sympy` for a few arithmetic functions and `mpmath` for high precision.
The sieves go up to $10^7$ (and $10^8$ for the gaps and digits); the whole notebook runs in a few minutes and needs about 1 GB of memory.

*Before running a cell, predict its output.*

In [ ]:
import math, random
from math import gcd, log, sqrt, pi, exp
import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
from sympy import mobius, divisors, divisor_count, totient, sqrt_mod, primerange
mp.mp.dps = 30
G = float(mp.euler)
rng = np.random.default_rng(13)

def sieve(N):
    s = np.ones(N + 1, bool); s[:2] = False
    for i in range(2, int(N ** .5) + 1):
        if s[i]: s[i * i::i] = False
    return s

N = 10 ** 7
S = sieve(N); P = np.nonzero(S)[0]
print(len(P), "primes below", N)

## Part A: checks

**Problem 13.1 (sums of primes).** $\sum_{p\le x}p$ against $\frac{x^2}{2\log x}$ and $\operatorname{Li}(x^2)$; the least prime factors; the sum of the first $n$ primes.

In [ ]:
x = N
Sp = int(P.astype(np.int64).sum())
print(Sp, Sp / (x * x / (2 * log(x))), Sp / float(mp.li(mp.mpf(x) ** 2) - mp.li(2)))
spf = np.zeros(N + 1, np.int64)
for p in P[::-1]: spf[p::p] = p
Sm = int(spf[2:].sum())
print("sum of P^-(n):", Sm, Sm / Sp, (Sm - Sp) / x ** 1.5)
n = len(P); print("first n primes:", Sp / (0.5 * n * n * log(n)))

**Problem 13.2 (largest prime factor).** $\sum_{n\le x}P^+(n)$ and $\sum_{n\le x}A(n)$ against $\frac{\pi^2}{12}\frac{x^2}{\log x}$, and the bound of (d).

In [ ]:
lpf = np.zeros(N + 1, np.int64)
for p in P: lpf[p::p] = p
SL = int(lpf[2:].sum())
A = 0
for p in P:
    q = int(p)
    while q <= x:
        A += int(p) * (x // q); q *= int(p)
print(SL, SL / (pi ** 2 * x * x / (12 * log(x))), A / (pi ** 2 * x * x / (12 * log(x))), (A - SL) / (x ** 1.5 * math.log2(x)))

**Problem 13.3 (radical).** $\frac2{x^2}\sum_{n\le x}\operatorname{rad}(n)$ against $C=\prod_p(1-\frac1{p(p+1)})$.

In [ ]:
rad = np.ones(N + 1, np.int64)
for p in P: rad[p::p] *= p
Cr = float(np.prod(1 - 1 / (P * (P + 1.0))))
R = int(rad[1:].sum())
print(2 * R / x ** 2, Cr, (R - Cr * x * x / 2) / x ** 1.5)

**Problem 13.4 (divisors of $n!$).** $\log\tau(n!)$ divided by $\frac n{\log n}$, against $C=\sum_k\frac1k\log(1+\frac1k)$ and $e^C$.

In [ ]:
Cf = mp.nsum(lambda k: mp.log(1 + 1 / k) / k, [1, mp.inf]); print(Cf, mp.e ** Cf)
for n in [10 ** 4, 10 ** 5, 10 ** 6, 10 ** 7]:
    L = 0.0
    for p in P[P <= n]:
        p = int(p); v = 0; q = p
        while q <= n: v += n // q; q *= p
        L += log(1 + v)
    print(n, round(L / (n / log(n)), 4))

**Problem 13.5 ($\omega(\binom{2n}n)$).** A prime divides $\binom{2n}n$ when adding $n+n$ in base $p$ has a carry (Kummer).

In [ ]:
def omega_central(n):
    c = 0
    for p in P[P <= 2 * n]:
        p = int(p); m = n
        while m:
            if 2 * (m % p) >= p: c += 1; break
            m //= p
    return c
for n in [10 ** 4, 10 ** 5, 10 ** 6, 5 * 10 ** 6]:
    c = omega_central(n); print(n, c, round(c / (log(4) * n / log(n)), 4))

**Problem 13.6 (prime zeta).** $P(s)=\sum_k\frac{\mu(k)}k\log\zeta(ks)$, truncated at $K$, against the direct sum over $p\le10^7$.

In [ ]:
def Pz(s, K):
    return sum(mobius(k) / mp.mpf(k) * mp.log(mp.zeta(k * s)) for k in range(1, K + 1) if mobius(k) != 0)
for K in [5, 10, 20, 40]: print(K, Pz(2, K))
direct = mp.fsum(1 / mp.mpf(int(p)) ** 2 for p in P)
print("direct:", direct, " difference:", Pz(2, 40) - direct, " 1/(x log x):", 1 / (x * log(x)))
print("P(3) =", Pz(3, 40))

**Problem 13.7 (primes with a prime index).** $\pi(\pi(x))$ and the partial sum of $\frac1{p_{p_n}}$.

In [ ]:
M8 = 10 ** 8
S8 = sieve(M8); P8 = np.nonzero(S8)[0]
for X in [10 ** 6, 10 ** 7, 10 ** 8]:
    k = np.searchsorted(P8, X, side='right'); kk = np.searchsorted(P8, k, side='right')
    print(X, k, kk, round(kk / (X / log(X) ** 2), 4), round(k / log(k) * (1 + 1 / log(k))))
SP = P8[P8[P8 < len(P8)] - 1]
print(SP[:8], (1 / SP.astype(float)).sum())

**Problem 13.8 (Ramanujan primes).** $R_n=1+\max\{x:f(x)<n\}$ over the integers, with $f(x)=\pi(x)-\pi(x/2)$; we check up to $2\cdot10^6$, enough for $n\le20000$.

In [ ]:
Mr = 2 * 10 ** 6
pic = np.cumsum(S8[:Mr + 1]); f = pic - pic[np.arange(Mr + 1) // 2]
nmax = 20000
last = np.full(nmax + 1, -1)
for xv in range(Mr + 1):
    if f[xv] <= nmax: last[f[xv]] = xv
Rn = np.zeros(nmax + 1, np.int64); run = -1
for k in range(1, nmax + 1):
    run = max(run, last[k - 1]); Rn[k] = run + 1
print(Rn[1:11])
for k in [10, 100, 1000, 10000, 20000]:
    p2 = P8[2 * k - 1]; print(k, Rn[k], p2, round(Rn[k] / p2, 4), round(1 + log(2) / log(p2), 4), Rn[k] < P8[3 * k - 1])

**Problems 13.9 and 13.10 ($p-1$).** Squarefree $p-1$, the means of $\frac{\varphi(p-1)}{p-1}$ and $\frac{p-1}{\varphi(p-1)}$, and $\sum\tau(p-1)$; then the primes for which $2$ is a primitive root.

In [ ]:
sqf = np.ones(N + 1, bool)
for p in P[P * P <= N]: sqf[p * p::p * p] = False
phi = np.arange(N + 1, dtype=np.float64)
for p in P: phi[p::p] *= 1 - 1 / p
tau = np.zeros(N + 1, np.int32)
for d in range(1, N + 1): tau[d::d] += 1
Art = float(np.prod(1 - 1 / (P * (P - 1.0))))
B = float(np.prod(1 + 1 / (P - 1.0) ** 2))
Z = float(mp.zeta(2) * mp.zeta(3) / mp.zeta(6))
print("A =", Art, " B =", B, " Z =", Z)
for X in [10 ** 5, 10 ** 6, 10 ** 7]:
    m = P[P <= X] - 1
    print(X, sqf[m].mean(), (phi[m] / m).mean(), (m / phi[m]).mean(), tau[m].sum() / X)
cnt = 0
for p in P[1:]:
    p = int(p); m = p - 1; ok = True
    while m > 1:
        q = int(spf[m])
        if pow(2, (p - 1) // q, p) == 1: ok = False; break
        while m % q == 0: m //= q
    cnt += ok
print("2 is a primitive root for", cnt, "of", len(P) - 1, "odd primes:", cnt / (len(P) - 1))

**Problem 13.11 (roots of $x^n-1$).** Menon's identity, and the average of $\gcd(n,p-1)$.

In [ ]:
for n in range(1, 400):
    assert sum(gcd(a - 1, n) for a in range(1, n + 1) if gcd(a, n) == 1) == totient(n) * divisor_count(n)
print("Menon's identity holds for n < 400")
Q6 = P[(P > 60) & (P < 10 ** 6)]
for n in [12, 24, 60]:
    print(n, np.mean([gcd(n, int(p) - 1) for p in Q6]), divisor_count(n))

**Problem 13.12 ($\omega(n^2+1)$).** Sieve $n^2+1$ by the primes $p\le x$ with $p=2$ or $p\equiv1\pmod4$; the cofactor left over is $1$ or a prime larger than $x$.

In [ ]:
X = 10 ** 6
nn = np.arange(1, X + 1, dtype=np.int64); rem = nn * nn + 1; om = np.zeros(X, np.int64)
for p in P[P <= X]:
    p = int(p)
    if p == 2: roots = [1]
    elif p % 4 == 1:
        r = int(sqrt_mod(-1, p)); roots = [r, p - r]
    else: continue
    for r in roots:
        idx = np.arange((r - 1) % p, X, p)
        om[idx] += 1
        while True:
            hit = rem[idx] % p == 0
            if not hit.any(): break
            rem[idx[hit]] //= p
om += rem > 1
for X1 in [10 ** 4, 10 ** 5, 10 ** 6]:
    o = om[:X1]; print(X1, o.mean() - log(log(X1)), o.var(), log(log(X1)))

**Problem 13.13 (Rankin).** $\Psi(x,y)$ by the recursion $\Psi(x,p_k)=\Psi(x,p_{k-1})+\Psi(\lfloor x/p_k\rfloor,p_k)$, against the best bound $x^\sigma\prod_{p\le y}(1-p^{-\sigma})^{-1}$.

In [ ]:
from functools import lru_cache
import bisect, sys
sys.setrecursionlimit(10000)
ps = list(primerange(2, 1000))
@lru_cache(None)
def Psi(X, k):
    if k == 0 or X < 2: return 1 if X >= 1 else 0
    if ps[k - 1] > X: return Psi(X, bisect.bisect_right(ps, X, 0, k))
    return Psi(X, k - 1) + Psi(X // ps[k - 1], k)
for X, y in [(10 ** 10, 100), (10 ** 12, 1000)]:
    k = bisect.bisect_right(ps, y); v = Psi(X, k)
    best = min(X ** s * np.prod([1 / (1 - p ** (-s)) for p in ps[:k]]) for s in np.linspace(0.01, 1, 400))
    print(X, y, v, "%.3g" % best, round(best / v, 1))

**Problems 13.14 and 13.15 (Dickman and Buchstab).** $\rho$ from $u\rho'(u)=-\rho(u-1)$, its integral against $e^\gamma$; $\omega$ from Buchstab's equation, against $e^{-\gamma}$; rough numbers up to $10^8$.

In [ ]:
h = 1e-4; U = 20; m = int(round(1 / h)); n = int(U / h)
rho = np.ones(n + 1)
for i in range(m, n):
    u = i * h
    rho[i + 1] = rho[i] - h * (rho[i - m] / u + rho[i + 1 - m] / (u + h)) / 2
print("int rho =", np.trapezoid(rho, dx=h), " e^gamma =", exp(G), " rho(2) =", rho[2 * m], 1 - log(2))
us = np.arange(1, 8 + h / 2, h); om_ = np.zeros(len(us)); cum = np.zeros(len(us))
for i, u in enumerate(us):
    om_[i] = 1 / u if u <= 2 + 1e-12 else (1 + cum[i - m]) / u
    if i: cum[i] = cum[i - 1] + h * (om_[i] + om_[i - 1]) / 2
print("omega(3) =", om_[2 * m], (1 + log(2)) / 3, " e^-gamma =", exp(-G))
print("max |omega - e^-gamma| on [4,8]:", np.abs(om_[3 * m:] - exp(-G)).max())
for u in (1.5, 2, 2.5, 3, 4):
    y = M8 ** (1 / u); a = np.ones(M8 + 1, bool); a[0] = False
    for p in P8[P8 <= y]: a[p::p] = False
    print(u, a.sum() * log(y) / M8)
del a

**Problem 13.16 (Mertens for polynomials).** $ne^\gamma E_n$ for $q=2,3,5$, and the half-product against $\frac{2e^{-\gamma}}n$.

In [ ]:
mp.mp.dps = 120
def Nk(k, q): return sum(mobius(d) * q ** (k // d) for d in divisors(k)) // k
for q in [2, 3, 5]:
    for n in [10, 20, 40, 80]:
        L = mp.fsum(Nk(k, q) * mp.log(1 - mp.mpf(q) ** (-k)) for k in range(1, n + 1))
        H = mp.fsum(Nk(k, q) * mp.log(1 - mp.mpf(q) ** (-k)) for k in range(1, n // 2 + 1))
        print(q, n, mp.nstr(mp.exp(L) * n * mp.exp(mp.euler), 7), mp.nstr(mp.exp(H) * n, 6), mp.nstr(2 * mp.exp(-mp.euler), 6))
mp.mp.dps = 30

**Problem 13.17 (Hawkins).** The recurrence $q_{n+1}=q_n-\frac{q_n^2}n$.

In [ ]:
qn = 1.0
for n in range(2, 10 ** 7 + 1):
    if n in (10 ** 3, 10 ** 5, 10 ** 7): print(n, qn * log(n), 1 / qn - log(n))
    qn -= qn * qn / n

**Problem 13.18 (Rényi's law).** The coefficients of $R(z)$, against the proportions among $n\le10^7$, and $2^kd_k$ against $\frac1{4C_2}$.

In [ ]:
Pl = [int(p) for p in P[:200000]]
def Rz(z):
    r = mp.mpf(1)
    for p in Pl: r *= 1 + (z - 1) / (p * (p - z))
    return r
d = mp.taylor(Rz, 0, 6)
diff = np.zeros(N + 1, np.int8)
for p in P[P * P <= N]:
    q = int(p) ** 2
    while q <= N: diff[q::q] += 1; q *= int(p)
C2 = mp.fprod(1 - 1 / mp.mpf(p - 1) ** 2 for p in Pl[1:])
for k in range(7):
    print(k, mp.nstr(d[k], 7), (diff[1:] == k).mean(), mp.nstr(2 ** k * d[k], 6))
print("1/(4 C2) =", 1 / (4 * C2), " mean:", sum(1 / (p * (p - 1.0)) for p in Pl), diff[1:].mean())

**Problem 13.19 (prime gaps).** $\frac1{x\log x}\sum_{p_{n+1}\le x}g_n^2$.

In [ ]:
g = np.diff(P8).astype(float); cs = np.cumsum(g * g)
for X in [10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6, 10 ** 7, 10 ** 8]:
    k = np.searchsorted(P8[1:], X, side='right'); print(X, cs[k - 1] / (X * log(X)))

**Problem 13.20 (last digits).** The transition frequencies after a prime ending in $1$, and the coin model with $q=\frac{2.5}{\log p}$.

In [ ]:
Q8 = P8[P8 > 5]; dg = Q8 % 10; a, b = dg[:-1], dg[1:]
row = np.array([((a == 1) & (b == t)).sum() for t in (1, 3, 7, 9)], float); print("data  1->1,3,7,9:", np.round(row / row.sum(), 4))
q = 2.5 / np.log(Q8[:-1][a == 1].astype(float)); den = 1 - (1 - q) ** 4
print("model 1->1,3,7,9:", np.round([np.mean(q * (1 - q) ** 3 / den), np.mean(q / den), np.mean(q * (1 - q) / den), np.mean(q * (1 - q) ** 2 / den)], 4))

## Part B: laboratory

**L13.1 (Hawkins primes by simulation).** Run Hawkins's sieve up to $10^5$ ten times. Compare the number of Hawkins primes with $\pi(10^5)=9592$, with $\sum_{n\le10^5}q_n$ and
with $\operatorname{Li}(10^5)$. How many of them are even?

**L13.2 (Buchstab's oscillation).** Compute $\omega(u)-e^{-\gamma}$ on $[2,10]$ and locate its sign changes. How fast does $\max_{[k,k+1]}|\omega-e^{-\gamma}|$ decrease? Compare with $\rho(k)$.

**L13.3 (a better coin model).** Redo the model of Problem 13.20 on the integers prime to $30$ instead of $10$, with $q=\frac{30}{8\log p}$, and recompute the four transitions
after a prime ending in $1$. Does the transition $1\to7$ come out right now? Why should it?

**L13.4 (all sixteen transitions).** Tabulate the $4\times4$ matrix of transitions between last digits for the primes below $10^8$. Check that it is close to symmetric under
$a\to b$ reversed and $a\mapsto10-a$ (that is $1\leftrightarrow9$, $3\leftrightarrow7$). Explain the second symmetry.

## Solutions

### L13.1

In [ ]:
def hawkins(X, rng):
    alive = np.ones(X + 1, bool); alive[:2] = False
    cur, out = 2, []
    while True:
        out.append(cur)
        rest = np.nonzero(alive[cur + 1:])[0] + cur + 1
        if len(rest) == 0: break
        alive[rest[rng.random(len(rest)) < 1 / cur]] = False
        nxt = np.nonzero(alive[cur + 1:])[0]
        if len(nxt) == 0: break
        cur = cur + 1 + nxt[0]
    return out
X = 10 ** 5
qs, qn = [], 1.0
for n in range(2, X + 1): qs.append(qn); qn -= qn * qn / n
runs = [hawkins(X, rng) for _ in range(10)]
print([len(r) for r in runs], np.mean([len(r) for r in runs]))
print("sum q_n =", sum(qs), " Li =", float(mp.li(X) - mp.li(2)), " even fraction:", np.mean([np.mean(np.array(r) % 2 == 0) for r in runs]))

The counts fluctuate widely, from about $6500$ to $9600$ in our runs: the first few Hawkins primes decide how hard the sieve is for everything after them. Their mean, about $8200$,
is below $\sum_{n\le10^5}q_n=8588$, as Problem 13.17(b) requires, and both are below $\pi(10^5)=9592$ and $\operatorname{Li}(10^5)=9629$, because $\frac1{q_n}=\log n+1.24+o(1)$ rather than
$\log n$. About half the Hawkins primes are even: the coins know nothing of divisibility.

### L13.2

In [ ]:
us2 = np.arange(1, 10 + 1e-3 / 2, 1e-3); w = np.zeros(len(us2)); cw = np.zeros(len(us2)); mm = 1000
for i, u in enumerate(us2):
    w[i] = 1 / u if u <= 2 + 1e-12 else (1 + cw[i - mm]) / u
    if i: cw[i] = cw[i - 1] + 1e-3 * (w[i] + w[i - 1]) / 2
dlt = w - exp(-G)
sc = us2[1:][np.sign(dlt[1:]) != np.sign(dlt[:-1])]
print("sign changes near u =", np.round(sc[sc > 2], 2))
for k in range(2, 9):
    sel = (us2 >= k) & (us2 <= k + 1)
    print(k, "%.2e" % np.abs(dlt[sel]).max(), "%.2e" % rho[int(k / h)])

The difference changes sign at intervals a little shorter than $1$ (near $2.48$, $3.23$, $4.00$, $4.79$, $5.56$, \dots), and its maximum on $[k,k+1]$ falls faster than any exponential,
at least as fast as $\rho(k)$. From about $u=7$ on it reaches the accuracy of the grid ($10^{-3}$ steps with the trapezoidal rule), so the last sign changes and the last maxima should not be
trusted.

### L13.3

In [ ]:
R30 = [r for r in range(30) if gcd(r, 30) == 1]
def trans30(q, i):
    t = np.zeros(8)
    for j in range(1, 8 * 80): t[(i + j) % 8] += q * (1 - q) ** (j - 1)
    return t
ends1 = Q8[:-1][a == 1]; res = np.zeros(4); tot = 0; pos = {1: 0, 3: 1, 7: 2, 9: 3}
qq = np.round((30 / 8) / np.log(ends1.astype(float)), 3)
for r in (1, 11):
    sel = ends1 % 30 == r; vals, cnts = np.unique(qq[sel], return_counts=True)
    for qv, c in zip(vals, cnts):
        t = trans30(qv, R30.index(r))
        for jj in range(8): res[pos[R30[jj] % 10]] += c * t[jj]
        tot += c
print("mod-30 model 1->1,3,7,9:", np.round(res / tot, 4)); print("data              :", np.round(row / row.sum(), 4))

The model on integers prime to $30$ gives about $18.2\%$, $30.2\%$, $31.5\%$, $20.1\%$, against $17.7\%$, $30.4\%$, $31.0\%$, $20.9\%$ for the primes: the transition $1\to7$ is now
right to within half a point. Knowing about $3$ is enough to see that the gap $6$ is twice as likely as $2$ or $4$: in the walk through residues prime to $30$, a number $\equiv1\pmod{30}$
is followed by $7$ at distance $6$ with no residue in between, whereas from $11$ the next candidate is $13$. The remaining discrepancy comes from $7$, $11$, $13,\dots$, which the model
still ignores, and from the size of $q$.

### L13.4

In [ ]:
D = np.array([[((a == s) & (b == t)).sum() for t in (1, 3, 7, 9)] for s in (1, 3, 7, 9)], float)
Dn = D / D.sum(axis=1, keepdims=True)
print(np.round(Dn, 4))
perm = [3, 2, 1, 0]   # 1<->9, 3<->7
print("max |D[s,t] - D[10-t,10-s]| / total:", np.abs(D - D[perm][:, perm].T).max() / D.sum())

The counts satisfy $D(s,t)\approx D(10-t,10-s)$: a gap from a prime ending in $s$ to one ending in $t$ is the same as a gap of the same length from $-t$ to $-s$, and reflecting the integers
$n\mapsto-n$ (modulo $10$) maps the endings $1,3,7,9$ to $9,7,3,1$ and reverses the order. The prime number theorem in progressions makes the primes on average indifferent to this
reflection, and the Hardy--Littlewood conjectures predict that the pair statistics are too, which is what the table shows to about one part in a thousand.